# Distance group comparisons

# Setting up

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import glob
import os
import json
from scipy.stats import ttest_ind, mannwhitneyu
from scipy.stats import linregress
from collections import defaultdict

mkdir -p failed for path /gpfs/home/yb2612/.config/matplotlib: [Errno 13] Permission denied: '/gpfs/home/yb2612/.config/matplotlib'
Matplotlib created a temporary cache directory at /tmp/matplotlib-bw9rx2bg because there was an issue with the default path (/gpfs/home/yb2612/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


# Fixing metadata

In [2]:
# load total metadata fixed
total_metadata = pd.read_csv("/gpfs/home/yb2612/yb2612_fenyo/data/seurat_objects/Cervical_v5_obj_metadata_total_metadata_radial_distances_kmeans.csv")

/tmp/ipykernel_518772/2287263937.py:2: DtypeWarning: Columns (9,16,73) have mixed types. Specify dtype option on import or set low_memory=False.
  total_metadata = pd.read_csv("/gpfs/home/yb2612/yb2612_fenyo/data/seurat_objects/Cervical_v5_obj_metadata_total_metadata_radial_distances_kmeans.csv")


In [81]:
print(total_metadata.shape)
print(total_metadata.columns)
total_metadata

(6415279, 76)
Index(['orig.ident', 'nCount_CODEX', 'nFeature_CODEX', 'absolute_x',
       'absolute_y', 'slide_id', 'Age_at_Diagnosis', 'Stage_at_diagnosis',
       'CPS_Score', 'Specimen_from_Path', 'Radiation_Therapy', 'Response',
       'Patient_deceased', 'Subsequent_Recurrence', 'Recurrence_number_2',
       'Maintainence_regimen', 'Treatment_after_Immunotherapy',
       'Length_of_Treatment_days', 'Diagnosis_Year', 'Final.cell.type',
       'immune', 'Fine.cell.type', 'r_dist_Tumor', 'r_dist_Cycling_Tumor',
       'r_dist_CD8_T', 'r_dist_Macrophage_CD163pos', 'r_dist_Activated_CD8_T',
       'r_dist_Stromal_Undefined_Unknown', 'r_dist_Cytotoxic_NK',
       'r_dist_Neutrophil', 'r_dist_B', 'r_dist_Treg', 'r_dist_Activated_T',
       'r_dist_Exhausted_CD8', 'r_dist_Activated_Cytotoxic_NK', 'r_dist_T',
       'r_dist_CD4_T', 'r_dist_Activated_CD4_T', 'r_dist_Th1',
       'r_dist_Activated_B', 'r_dist_Activated_Exhausted_CD8',
       'r_dist_Endothelial', 'r_dist_Activated_Neutrophil

,orig.ident,nCount_CODEX,nFeature_CODEX,absolute_x,absolute_y,slide_id,Age_at_Diagnosis,Stage_at_diagnosis,CPS_Score,Specimen_from_Path,...,Activated_Exhausted_CD8_dist_kmeans,Endothelial_dist_kmeans,Activated_Neutrophil_dist_kmeans,Activated_Treg_dist_kmeans,Macrophage_CD163neg_dist_kmeans,Activated_Th1_dist_kmeans,Activated_Macrophage_CD163pos_dist_kmeans,Activated_Macrophage_CD163neg_dist_kmeans,Parent.cell.type,Grandparent.cell.type
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
02433_Cell_1,02433,19169.929159,34,6404.047059,3699.258824,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Peri,Far,Peri,Peri,Peri,Peri,Far,Far,Tumor,Tumor
02433_Cell_2,02433,20208.610168,34,6403.780000,3708.830000,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Peri,Far,Peri,Peri,Peri,Peri,Far,Far,Tumor,Tumor
02433_Cell_3,02433,22242.868252,34,6402.186813,3726.461538,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Near,Far,Peri,Peri,Peri,Peri,Far,Far,Tumor,Tumor
02433_Cell_4,02433,22799.356619,33,6402.834783,3744.017391,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Near,Far,Peri,Peri,Peri,Peri,Far,Far,Tumor,Tumor
02433_Cell_5,02433,9322.615338,34,6401.584615,3820.276923,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Near,Far,Peri,Near,Peri,Near,Peri,Far,CD8_T,T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28873_Cell_26050,28873,1050.076929,30,2299.346154,15099.346154,20250318-Jharna-28873-A1_Scan1.er,42,IIICir,50,Cervical biopsy,...,Far,Peri,Peri,Peri,Far,Far,Far,Far,Stromal_Undefined_Unknown,Stromal_Undefined_Unknown
28873_Cell_26051,28873,12199.701474,30,2300.912281,14937.754386,20250318-Jharna-28873-A1_Scan1.er,42,IIICir,50,Cervical biopsy,...,Far,Peri,Peri,Near,Far,Far,Far,Far,Tumor,Tumor
28873_Cell_26052,28873,12423.254193,33,2299.704663,14975.445596,20250318-Jharna-28873-A1_Scan1.er,42,IIICir,50,Cervical biopsy,...,Far,Peri,Peri,Near,Far,Far,Far,Far,Tumor,Tumor


## Creating parent/grandparent cell types

In [4]:
# Create Parent.cell.type with replacements
total_metadata["Parent.cell.type"] = total_metadata["Final.cell.type"]

# Replace based on conditions
total_metadata["Parent.cell.type"] = np.where(
    total_metadata["Parent.cell.type"].str.contains("Treg|Th1", case=True, na=False),
    "CD4_T",
    total_metadata["Parent.cell.type"]
)
total_metadata["Parent.cell.type"] = np.where(
    total_metadata["Parent.cell.type"].str.contains("NK|Exhausted", case=True, na=False),
    "CD8_T",
    total_metadata["Parent.cell.type"]
)
total_metadata["Parent.cell.type"] = np.where(
    total_metadata["Parent.cell.type"].str.contains("Macrophage", case=True, na=False),
    "Macrophage",
    total_metadata["Parent.cell.type"]
)

total_metadata["Parent.cell.type"] = np.where(
    total_metadata["Parent.cell.type"].str.contains("Undefined|Unknown", case=True, na=False),
    "Stromal_Undefined_Unknown",
    total_metadata["Parent.cell.type"]
)

# Create Grandparent.cell.type from Parent.cell.type
total_metadata["Grandparent.cell.type"] = np.where(
    total_metadata["Parent.cell.type"].isin(["CD4_T", "CD8_T"]),
    "T",
    total_metadata["Parent.cell.type"]
)

total_metadata[["Fine.cell.type", "Parent.cell.type", "Grandparent.cell.type"]]

,Fine.cell.type,Parent.cell.type,Grandparent.cell.type
0,Tumor,Tumor,Tumor
1,Cycling_Tumor,Tumor,Tumor
2,Cycling_Tumor,Tumor,Tumor
3,Cycling_Tumor,Tumor,Tumor
4,CD8_T,CD8_T,T
...,...,...,...
6415274,Stromal_Undefined_Unknown,Stromal_Undefined_Unknown,Stromal_Undefined_Unknown
6415275,Tumor,Tumor,Tumor
6415276,Tumor,Tumor,Tumor
6415277,T,T,T


In [5]:
# fixed coords
total_metadata['Grandparent.cell.type'].value_counts()

Grandparent.cell.type
Tumor                        2892549
Stromal_Undefined_Unknown    1207143
T                            1013277
Macrophage                    593416
Endothelial                   344890
Neutrophil                    259351
B                             104653
Name: count, dtype: int64

In [8]:
cell_to_parent = dict(zip(total_metadata['Fine.cell.type'], total_metadata['Parent.cell.type']))
cell_to_grandparent = dict(zip(total_metadata['Fine.cell.type'], total_metadata['Grandparent.cell.type']))
print(cell_to_parent)
print(cell_to_grandparent)

{'Tumor': 'Tumor', 'Cycling_Tumor': 'Tumor', 'CD8_T': 'CD8_T', 'Macrophage_CD163pos': 'Macrophage', 'Activated_CD8_T': 'CD8_T', 'Stromal_Undefined_Unknown': 'Stromal_Undefined_Unknown', 'Cytotoxic_NK': 'CD8_T', 'Neutrophil': 'Neutrophil', 'B': 'B', 'Treg': 'CD4_T', 'Activated_T': 'T', 'Exhausted_CD8': 'CD8_T', 'Activated_Cytotoxic_NK': 'CD8_T', 'T': 'T', 'CD4_T': 'CD4_T', 'Activated_CD4_T': 'CD4_T', 'Th1': 'CD4_T', 'Activated_B': 'B', 'Activated_Exhausted_CD8': 'CD8_T', 'Endothelial': 'Endothelial', 'Activated_Neutrophil': 'Neutrophil', 'Activated_Treg': 'CD4_T', 'Macrophage_CD163neg': 'Macrophage', 'Activated_Th1': 'CD4_T', 'Activated_Macrophage_CD163pos': 'Macrophage', 'Activated_Macrophage_CD163neg': 'Macrophage'}
{'Tumor': 'Tumor', 'Cycling_Tumor': 'Tumor', 'CD8_T': 'T', 'Macrophage_CD163pos': 'Macrophage', 'Activated_CD8_T': 'T', 'Stromal_Undefined_Unknown': 'Stromal_Undefined_Unknown', 'Cytotoxic_NK': 'T', 'Neutrophil': 'Neutrophil', 'B': 'B', 'Treg': 'T', 'Activated_T': 'T', 'Ex

In [10]:
total_metadata = total_metadata.set_index("Unnamed: 0")
total_metadata

,orig.ident,nCount_CODEX,nFeature_CODEX,absolute_x,absolute_y,slide_id,Age_at_Diagnosis,Stage_at_diagnosis,CPS_Score,Specimen_from_Path,...,Activated_Exhausted_CD8_dist_kmeans,Endothelial_dist_kmeans,Activated_Neutrophil_dist_kmeans,Activated_Treg_dist_kmeans,Macrophage_CD163neg_dist_kmeans,Activated_Th1_dist_kmeans,Activated_Macrophage_CD163pos_dist_kmeans,Activated_Macrophage_CD163neg_dist_kmeans,Parent.cell.type,Grandparent.cell.type
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
02433_Cell_1,2433,19169.929159,34,6404.047059,3699.258824,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Peri,Far,Peri,Peri,Peri,Peri,Far,Far,Tumor,Tumor
02433_Cell_2,2433,20208.610168,34,6403.780000,3708.830000,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Peri,Far,Peri,Peri,Peri,Peri,Far,Far,Tumor,Tumor
02433_Cell_3,2433,22242.868252,34,6402.186813,3726.461538,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Near,Far,Peri,Peri,Peri,Peri,Far,Far,Tumor,Tumor
02433_Cell_4,2433,22799.356619,33,6402.834783,3744.017391,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Near,Far,Peri,Peri,Peri,Peri,Far,Far,Tumor,Tumor
02433_Cell_5,2433,9322.615338,34,6401.584615,3820.276923,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Near,Far,Peri,Near,Peri,Near,Peri,Far,CD8_T,T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28873_Cell_26050,28873,1050.076929,30,2299.346154,15099.346154,20250318-Jharna-28873-A1_Scan1.er,42,IIICir,50,Cervical biopsy,...,Far,Peri,Peri,Peri,Far,Far,Far,Far,Stromal_Undefined_Unknown,Stromal_Undefined_Unknown
28873_Cell_26051,28873,12199.701474,30,2300.912281,14937.754386,20250318-Jharna-28873-A1_Scan1.er,42,IIICir,50,Cervical biopsy,...,Far,Peri,Peri,Near,Far,Far,Far,Far,Tumor,Tumor
28873_Cell_26052,28873,12423.254193,33,2299.704663,14975.445596,20250318-Jharna-28873-A1_Scan1.er,42,IIICir,50,Cervical biopsy,...,Far,Peri,Peri,Near,Far,Far,Far,Far,Tumor,Tumor


In [82]:
total_metadata.to_csv("/gpfs/home/yb2612/yb2612_fenyo/data/seurat_objects/Cervical_v5_obj_metadata_total_metadata_radial_distances_kmeans_parent.csv", index=True)

## Map to tile

In [11]:
# total_metadata = pd.read_csv("/gpfs/home/yb2612/yb2612_fenyo/data/seurat_objects/intermediates/Cervical_v5_obj_metadata_total_metadata_radial_distances_kmeans_fixed_parent.csv")
total_metadata['orig.ident'] = total_metadata['orig.ident'].astype(str).str.zfill(5)  # pad orig ident with leading 0s to 5 digits
print(total_metadata.columns)
total_metadata.head()

Index(['orig.ident', 'nCount_CODEX', 'nFeature_CODEX', 'absolute_x',
       'absolute_y', 'slide_id', 'Age_at_Diagnosis', 'Stage_at_diagnosis',
       'CPS_Score', 'Specimen_from_Path', 'Radiation_Therapy', 'Response',
       'Patient_deceased', 'Subsequent_Recurrence', 'Recurrence_number_2',
       'Maintainence_regimen', 'Treatment_after_Immunotherapy',
       'Length_of_Treatment_days', 'Diagnosis_Year', 'Final.cell.type',
       'immune', 'Fine.cell.type', 'r_dist_Tumor', 'r_dist_Cycling_Tumor',
       'r_dist_CD8_T', 'r_dist_Macrophage_CD163pos', 'r_dist_Activated_CD8_T',
       'r_dist_Stromal_Undefined_Unknown', 'r_dist_Cytotoxic_NK',
       'r_dist_Neutrophil', 'r_dist_B', 'r_dist_Treg', 'r_dist_Activated_T',
       'r_dist_Exhausted_CD8', 'r_dist_Activated_Cytotoxic_NK', 'r_dist_T',
       'r_dist_CD4_T', 'r_dist_Activated_CD4_T', 'r_dist_Th1',
       'r_dist_Activated_B', 'r_dist_Activated_Exhausted_CD8',
       'r_dist_Endothelial', 'r_dist_Activated_Neutrophil',
       'r_d

,orig.ident,nCount_CODEX,nFeature_CODEX,absolute_x,absolute_y,slide_id,Age_at_Diagnosis,Stage_at_diagnosis,CPS_Score,Specimen_from_Path,...,Activated_Exhausted_CD8_dist_kmeans,Endothelial_dist_kmeans,Activated_Neutrophil_dist_kmeans,Activated_Treg_dist_kmeans,Macrophage_CD163neg_dist_kmeans,Activated_Th1_dist_kmeans,Activated_Macrophage_CD163pos_dist_kmeans,Activated_Macrophage_CD163neg_dist_kmeans,Parent.cell.type,Grandparent.cell.type
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
02433_Cell_1,02433,19169.929159,34,6404.047059,3699.258824,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Peri,Far,Peri,Peri,Peri,Peri,Far,Far,Tumor,Tumor
02433_Cell_2,02433,20208.610168,34,6403.780000,3708.830000,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Peri,Far,Peri,Peri,Peri,Peri,Far,Far,Tumor,Tumor
02433_Cell_3,02433,22242.868252,34,6402.186813,3726.461538,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Near,Far,Peri,Peri,Peri,Peri,Far,Far,Tumor,Tumor
02433_Cell_4,02433,22799.356619,33,6402.834783,3744.017391,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Near,Far,Peri,Peri,Peri,Peri,Far,Far,Tumor,Tumor
02433_Cell_5,02433,9322.615338,34,6401.584615,3820.276923,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Near,Far,Peri,Near,Peri,Near,Peri,Far,CD8_T,T


In [ ]:
# parent folder
parent_folder = '/gpfs/home/yb2612/yb2612_fenyo/data/cervical_samples'

# find all *_metadata.csv files in subfolders
metadata_files = glob.glob(os.path.join(parent_folder, '**', '*_metadata_fixed.csv'), recursive=True)

# load, concat
df_list = [pd.read_csv(f) for f in metadata_files]
metadata_concat = pd.concat(df_list, ignore_index=True)

print(f"Loaded {len(metadata_files)} files. Combined shape: {metadata_concat.shape}")
print(metadata_concat.columns)
metadata_concat.head()

In [185]:
cols_to_add = ['centroid_x', 'centroid_y', 'tile_h', 'tile_w']

merged_pieces = []

for sample_id in total_metadata['orig.ident'].unique():
    tm_sample = total_metadata[total_metadata['orig.ident'] == sample_id].copy()
    mc_sample = metadata_concat[metadata_concat['slide_id'].str.contains(sample_id, na=False)].copy()

    if mc_sample.empty:
        print(f"Warning: No matching slide_id found for sample {sample_id}")
        continue

    # Sort by coordinates
    tm_sample_sorted = tm_sample.sort_values(by=['x', 'y']).reset_index(drop=True)
    mc_sample_sorted = mc_sample.sort_values(by=['absolute_x', 'absolute_y']).reset_index(drop=True)

    # Check if number of rows match
    if len(tm_sample_sorted) != len(mc_sample_sorted):
        print(f"Warning: Row count mismatch for sample {sample_id} ({len(tm_sample_sorted)} vs {len(mc_sample_sorted)})")

    # Compare coordinates row-wise (stop at min length to avoid index errors)
    min_len = min(len(tm_sample_sorted), len(mc_sample_sorted))
    mismatches = []
    for i in range(min_len):
        x1, y1 = tm_sample_sorted.at[i, 'x'], tm_sample_sorted.at[i, 'y']
        x2, y2 = mc_sample_sorted.at[i, 'absolute_x'], mc_sample_sorted.at[i, 'absolute_y']
        if (x1 != x2) or (y1 != y2):
            mismatches.append((i, x1, y1, x2, y2))

    if mismatches:
        print(f"Coordinate mismatches found in sample {sample_id}:")
        for idx, x1, y1, x2, y2 in mismatches:
            print(f" Row {idx}: total_metadata (x,y)=({x1},{y1}) != metadata_concat (absolute_x,absolute_y)=({x2},{y2})")
    else:
        print(f"All coordinates matched for sample {sample_id} after sorting.")

    # Add columns only if no mismatches or handle carefully if mismatches exist
    if not mismatches:
        for col in cols_to_add:
            tm_sample_sorted[col] = mc_sample_sorted[col].values
    else:
        print(f"Skipping column addition for sample {sample_id} due to mismatches.")

    merged_pieces.append(tm_sample_sorted)

total_metadata_merged = pd.concat(merged_pieces, ignore_index=True)

print(f"Final merged DataFrame shape: {total_metadata_merged.shape}")
total_metadata_merged.head()

All coordinates matched for sample 02433 after sorting.
All coordinates matched for sample 08153 after sorting.
All coordinates matched for sample 10103 after sorting.
All coordinates matched for sample 00862 after sorting.
All coordinates matched for sample 09002 after sorting.
All coordinates matched for sample 10285 after sorting.
All coordinates matched for sample 34933 after sorting.
All coordinates matched for sample 00438 after sorting.
All coordinates matched for sample 07688 after sorting.
All coordinates matched for sample 39367 after sorting.
All coordinates matched for sample 49411 after sorting.
All coordinates matched for sample 04738 after sorting.
All coordinates matched for sample 07291 after sorting.
All coordinates matched for sample 28873 after sorting.
Final merged DataFrame shape: (6415279, 85)


,X,Unnamed: 0,orig.ident,nCount_RNA,nFeature_RNA,Age_at_Diagnosis,Stage_at_diagnosis,CPS_Score,Specimen_from_Path,Radiation_Therapy,...,Macrophage_CD163neg_dist_kmeans,Activated_Th1_dist_kmeans,Activated_Macrophage_CD163neg_dist_kmeans,Activated_Macrophage_CD163pos_dist_kmeans,Parent.cell.type,Grandparent.cell.type,centroid_x,centroid_y,tile_h,tile_w
0,02433_Cell104196,104196,02433,12822.125000,34,54,IV,>1,Cervical biopsy,Yes,...,Moderately_near,Moderately_near,Near,Near,Macrophage,Macrophage,0.687500,165.250000,12544.0,1024.0
1,02433_Cell34765,34765,02433,10075.666669,34,54,IV,>1,Cervical biopsy,Yes,...,Near,Moderately_near,Moderately_near,Near,CD4_T,T,0.809524,209.000000,10240.0,1024.0
2,02433_Cell49960,49960,02433,2882.000049,29,54,IV,>1,Cervical biopsy,Yes,...,Near,Near,Near,Near,Stromal_Undefined_Unknown,Stromal_Undefined_Unknown,0.909091,46.500000,11008.0,1024.0
3,02433_Cell22080,22080,02433,4377.499977,34,54,IV,>1,Cervical biopsy,Yes,...,Moderately_near,Far,Moderately_near,Moderately_near,Stromal_Undefined_Unknown,Stromal_Undefined_Unknown,0.954545,150.818182,9216.0,1024.0
4,02433_Cell85101,85101,02433,17595.433251,33,54,IV,>1,Cervical biopsy,Yes,...,Moderately_near,Far,Near,Near,Tumor,Tumor,1.100000,210.333333,12032.0,1024.0


In [186]:
tile_ids = []

for sample_id, group in total_metadata_merged.groupby('orig.ident'):
    # Get unique (tile_h, tile_w) for this sample
    unique_tiles = group[['tile_h', 'tile_w']].drop_duplicates().reset_index(drop=True)

    # Assign a unique tile number for each unique tile
    tile_mapping = {
        (row.tile_h, row.tile_w): f"{sample_id}_Tile{idx + 1}"
        for idx, row in unique_tiles.iterrows()
    }

    print(f"Sample {sample_id} has {len(tile_mapping)} unique tiles.")

    # Map back to the original group rows
    group_tile_ids = group.apply(
        lambda row: tile_mapping.get((row.tile_h, row.tile_w)), axis=1
    )

    tile_ids.append(group_tile_ids)

# Combine all mapped tile_ids
total_metadata_merged['tile_id'] = pd.concat(tile_ids).reset_index(drop=True)


Sample 00438 has 1599 unique tiles.
Sample 00862 has 7814 unique tiles.
Sample 02433 has 821 unique tiles.
Sample 04738 has 3386 unique tiles.
Sample 07291 has 6259 unique tiles.
Sample 07688 has 1710 unique tiles.
Sample 08153 has 653 unique tiles.
Sample 09002 has 1636 unique tiles.
Sample 10103 has 113 unique tiles.
Sample 10285 has 14924 unique tiles.
Sample 28873 has 179 unique tiles.
Sample 34933 has 189 unique tiles.
Sample 39367 has 462 unique tiles.
Sample 49411 has 296 unique tiles.


In [187]:
# want these to be the first cols
first_cols = [
    'X',
    'orig.ident',
    'nCount_RNA',
    'nFeature_RNA',
    'Final.cell.type',
    'Fine.cell.type',  # combined into Stromal_Undefined_Unknown
    'Fine.cell.type2',  # both Stromal_Undefined and Unknown cols present
    'Parent.cell.type',  # Final.cell.type, Macrophage, CD4_T, CD8_T, T
    'Grandparent.cell.type',  # Final.cell.type, Macrophage, T
    'tile_id',  # unique tile identifier
    'x',
    'y',
    'centroid_x',
    'centroid_y',
    'tile_h',
    'tile_w',
    
]

# get all other columns, preserving original order
other_cols = [col for col in total_metadata_merged.columns if col not in first_cols]

# reorder
total_metadata_merged = total_metadata_merged[first_cols + other_cols][first_cols + other_cols]

print(total_metadata_merged.columns)
total_metadata_merged.head()

Index(['X', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'Final.cell.type',
       'Fine.cell.type', 'Fine.cell.type2', 'Parent.cell.type',
       'Grandparent.cell.type', 'tile_id', 'x', 'y', 'centroid_x',
       'centroid_y', 'tile_h', 'tile_w', 'Unnamed: 0', 'Age_at_Diagnosis',
       'Stage_at_diagnosis', 'CPS_Score', 'Specimen_from_Path',
       'Radiation_Therapy', 'Response', 'Patient_deceased',
       'Subsequent_Recurrence', 'Recurrence_number_2', 'Maintainence_regimen',
       'Treatment_after_Immunotherapy', 'Length_of_Treatment_days',
       'Diagnosis_Year', 'barcodes', 'RNA_snn_res.0.4', 'seurat_clusters',
       'immune', 'r_dist_Tumor', 'r_dist_Cycling_Tumor', 'r_dist_Th1',
       'r_dist_Macrophage_CD163pos', 'r_dist_Activated_B',
       'r_dist_Stromal_Undefined_Unknown', 'r_dist_Cytotoxic_NK',
       'r_dist_Neutrophil', 'r_dist_B', 'r_dist_Exhausted_CD8', 'r_dist_Treg',
       'r_dist_CD8_T', 'r_dist_Activated_T', 'r_dist_T',
       'r_dist_Activated_CD8_T', 'r_dist_

,X,orig.ident,nCount_RNA,nFeature_RNA,Final.cell.type,Fine.cell.type,Fine.cell.type2,Parent.cell.type,Grandparent.cell.type,tile_id,...,CD4_T_dist_kmeans,Activated_CD4_T_dist_kmeans,Activated_Exhausted_CD8_dist_kmeans,Endothelial_dist_kmeans,Activated_Neutrophil_dist_kmeans,Activated_Treg_dist_kmeans,Macrophage_CD163neg_dist_kmeans,Activated_Th1_dist_kmeans,Activated_Macrophage_CD163neg_dist_kmeans,Activated_Macrophage_CD163pos_dist_kmeans
0,02433_Cell104196,02433,12822.125000,34,Macrophage_CD163pos,Macrophage_CD163pos,Macrophage_CD163pos,Macrophage,Macrophage,00438_Tile1,...,Moderately_near,Near,Near,Moderately_near,Near,Moderately_near,Moderately_near,Moderately_near,Near,Near
1,02433_Cell34765,02433,10075.666669,34,Th1,Th1,Th1,CD4_T,T,00438_Tile2,...,Near,Far,Far,Near,Moderately_near,Far,Near,Moderately_near,Moderately_near,Near
2,02433_Cell49960,02433,2882.000049,29,Stromal_Undefined,Stromal_Undefined_Unknown,Stromal_Undefined,Stromal_Undefined_Unknown,Stromal_Undefined_Unknown,00438_Tile3,...,Far,Far,Far,Far,Near,Far,Near,Near,Near,Near
3,02433_Cell22080,02433,4377.499977,34,Stromal_Undefined,Stromal_Undefined_Unknown,Stromal_Undefined,Stromal_Undefined_Unknown,Stromal_Undefined_Unknown,00438_Tile4,...,Far,Far,Far,Near,Near,Far,Moderately_near,Far,Moderately_near,Moderately_near
4,02433_Cell85101,02433,17595.433251,33,Tumor,Tumor,Tumor,Tumor,Tumor,00438_Tile5,...,Far,Near,Far,Near,Near,Far,Moderately_near,Far,Near,Near


In [188]:
total_metadata_merged.to_csv("/gpfs/home/yb2612/yb2612_fenyo/data/seurat_objects/Cervical_v5_obj_metadata_total_metadata_radial_distances_kmeans_fixed_parent_tiles.csv", index=False, quoting=1)

## Normalizing

In [13]:
import pandas as pd
# total_metadata = pd.read_csv("/gpfs/home/yb2612/yb2612_fenyo/data/seurat_objects/Cervical_v5_obj_metadata_total_metadata_radial_distances_kmeans_fixed_parent_tiles_updated.csv")
print(total_metadata["orig.ident"][total_metadata["Response"] == "Yes"].unique())

['02433' '10103' '00862' '09002' '00438' '39367' '04738' '07291']


In [51]:
total_metadata

,orig.ident,nCount_CODEX,nFeature_CODEX,absolute_x,absolute_y,slide_id,Age_at_Diagnosis,Stage_at_diagnosis,CPS_Score,Specimen_from_Path,...,Activated_Exhausted_CD8_dist_kmeans,Endothelial_dist_kmeans,Activated_Neutrophil_dist_kmeans,Activated_Treg_dist_kmeans,Macrophage_CD163neg_dist_kmeans,Activated_Th1_dist_kmeans,Activated_Macrophage_CD163pos_dist_kmeans,Activated_Macrophage_CD163neg_dist_kmeans,Parent.cell.type,Grandparent.cell.type
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
02433_Cell_1,02433,19169.929159,34,6404.047059,3699.258824,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Peri,Far,Peri,Peri,Peri,Peri,Far,Far,Tumor,Tumor
02433_Cell_2,02433,20208.610168,34,6403.780000,3708.830000,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Peri,Far,Peri,Peri,Peri,Peri,Far,Far,Tumor,Tumor
02433_Cell_3,02433,22242.868252,34,6402.186813,3726.461538,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Near,Far,Peri,Peri,Peri,Peri,Far,Far,Tumor,Tumor
02433_Cell_4,02433,22799.356619,33,6402.834783,3744.017391,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Near,Far,Peri,Peri,Peri,Peri,Far,Far,Tumor,Tumor
02433_Cell_5,02433,9322.615338,34,6401.584615,3820.276923,20250225-Jharna-02433-A1_Scan1.er,54,IV,>1,Cervical biopsy,...,Near,Far,Peri,Near,Peri,Near,Peri,Far,CD8_T,T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28873_Cell_26050,28873,1050.076929,30,2299.346154,15099.346154,20250318-Jharna-28873-A1_Scan1.er,42,IIICir,50,Cervical biopsy,...,Far,Peri,Peri,Peri,Far,Far,Far,Far,Stromal_Undefined_Unknown,Stromal_Undefined_Unknown
28873_Cell_26051,28873,12199.701474,30,2300.912281,14937.754386,20250318-Jharna-28873-A1_Scan1.er,42,IIICir,50,Cervical biopsy,...,Far,Peri,Peri,Near,Far,Far,Far,Far,Tumor,Tumor
28873_Cell_26052,28873,12423.254193,33,2299.704663,14975.445596,20250318-Jharna-28873-A1_Scan1.er,42,IIICir,50,Cervical biopsy,...,Far,Peri,Peri,Near,Far,Far,Far,Far,Tumor,Tumor


In [15]:
fine_cell_types = total_metadata["Fine.cell.type"].unique()
# cols containing dist groups (near, moderately near, far)
dist_cols = {cell: f'{cell}_dist_kmeans' for cell in fine_cell_types}
dist_cols

{'Tumor': 'Tumor_dist_kmeans',
 'Cycling_Tumor': 'Cycling_Tumor_dist_kmeans',
 'CD8_T': 'CD8_T_dist_kmeans',
 'Macrophage_CD163pos': 'Macrophage_CD163pos_dist_kmeans',
 'Activated_CD8_T': 'Activated_CD8_T_dist_kmeans',
 'Stromal_Undefined_Unknown': 'Stromal_Undefined_Unknown_dist_kmeans',
 'Cytotoxic_NK': 'Cytotoxic_NK_dist_kmeans',
 'Neutrophil': 'Neutrophil_dist_kmeans',
 'B': 'B_dist_kmeans',
 'Treg': 'Treg_dist_kmeans',
 'Activated_T': 'Activated_T_dist_kmeans',
 'Exhausted_CD8': 'Exhausted_CD8_dist_kmeans',
 'Activated_Cytotoxic_NK': 'Activated_Cytotoxic_NK_dist_kmeans',
 'T': 'T_dist_kmeans',
 'CD4_T': 'CD4_T_dist_kmeans',
 'Activated_CD4_T': 'Activated_CD4_T_dist_kmeans',
 'Th1': 'Th1_dist_kmeans',
 'Activated_B': 'Activated_B_dist_kmeans',
 'Activated_Exhausted_CD8': 'Activated_Exhausted_CD8_dist_kmeans',
 'Endothelial': 'Endothelial_dist_kmeans',
 'Activated_Neutrophil': 'Activated_Neutrophil_dist_kmeans',
 'Activated_Treg': 'Activated_Treg_dist_kmeans',
 'Macrophage_CD163neg'

## Proportions

In [152]:
from functools import reduce

all_proportion_dfs = {}

# normalize_col = ["orig.ident"]
celltype_cols = ["Fine.cell.type","Final.cell.type","Parent.cell.type","Grandparent.cell.type"]
cell_types = total_metadata[celltype_col].unique()
    
# subset to sample id and cell type
for celltype_col in celltype_cols:
    print(celltype_col)
        
    subset = total_metadata[['orig.ident', celltype_col]]
    
    # count = count of cells grouped by sample and fine cell type
    counts = subset.groupby(['orig.ident', celltype_col]).size().reset_index(name='count')    
    
    # total_count = count of cells grouped by sample
    total_per_group = subset.groupby(['orig.ident']).size().reset_index(name='total_count')
    
    # merge counts and total_counts, then get pct
    merged = counts.merge(total_per_group, on=['orig.ident'])
    merged['pct'] = merged['count'] / merged['total_count'] * 100
    
    # Add the Response column to merged_df by merging on orig.ident
    response_df = total_metadata[['orig.ident', 'Response']].drop_duplicates(subset='orig.ident')
    merged = merged.merge(response_df, on='orig.ident', how='left')
    all_proportion_dfs[celltype_col] = merged

all_proportion_dfs

Fine.cell.type
Final.cell.type
Parent.cell.type
Grandparent.cell.type


{'Fine.cell.type':     orig.ident             Fine.cell.type  count  total_count        pct  \
 0        00438                Activated_B     97       264973   0.036608   
 1        00438            Activated_CD4_T   6402       264973   2.416095   
 2        00438            Activated_CD8_T   1473       264973   0.555906   
 3        00438     Activated_Cytotoxic_NK    881       264973   0.332487   
 4        00438    Activated_Exhausted_CD8    347       264973   0.130957   
 ..         ...                        ...    ...          ...        ...   
 358      49411  Stromal_Undefined_Unknown   5847        49632  11.780706   
 359      49411                          T     79        49632   0.159172   
 360      49411                        Th1     33        49632   0.066489   
 361      49411                       Treg     54        49632   0.108801   
 362      49411                      Tumor  23603        49632  47.556012   
 
     Response  
 0        Yes  
 1        Yes  
 2      

In [65]:
all_proportion_dfs["Grandparent.cell.type"]

,orig.ident,Grandparent.cell.type,count,total_count,pct,Response
0,00438,B,1184,264973,0.446838,Yes
1,00438,Endothelial,13857,264973,5.229589,Yes
2,00438,Macrophage,17423,264973,6.575387,Yes
3,00438,Neutrophil,25895,264973,9.772694,Yes
4,00438,Stromal_Undefined_Unknown,40861,264973,15.420816,Yes
...,...,...,...,...,...,...
93,49411,Macrophage,4114,49632,8.289007,No
94,49411,Neutrophil,1931,49632,3.890635,No
95,49411,Stromal_Undefined_Unknown,5847,49632,11.780706,No
96,49411,T,1159,49632,2.335187,No


In [154]:
import json 

def make_json_serializable(obj):
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient='records')
    elif isinstance(obj, dict):
        return {k: make_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_json_serializable(v) for v in obj]
    else:
        return obj

# convert all recursively
all_proportion_dfs_serializable = make_json_serializable(all_proportion_dfs)

# save
with open('/gpfs/home/yb2612/yb2612_fenyo/data/seurat_objects/plots/cell_proportion_plots/all_proportion_dfs.json', 'w') as f:
    json.dump(all_proportion_dfs_serializable, f, indent=4)

In [69]:
for celltype_col in celltype_cols:
    print("saving", celltype_col)
    all_proportion_dfs[celltype_col].to_csv(f'/gpfs/home/yb2612/yb2612_fenyo/data/seurat_objects/plots/cell_proportion_plots/{celltype_col}_proportions.csv', index=False)

saving Fine.cell.type
saving Final.cell.type
saving Parent.cell.type
saving Grandparent.cell.type


In [153]:
print("all_proportion_dfs")
for celltype_col, df in all_proportion_dfs.items():
    print(f"  └── {celltype_col:<25} → DataFrame shape: {df.shape}")

all_proportion_dfs
  └── Fine.cell.type            → DataFrame shape: (363, 6)
  └── Final.cell.type           → DataFrame shape: (210, 6)
  └── Parent.cell.type          → DataFrame shape: (126, 6)
  └── Grandparent.cell.type     → DataFrame shape: (98, 6)


# Distance group

In [110]:
from functools import reduce

# store all dfs in dict
all_pct_dfs = {}

# cols we want to normalize by (denominator)
normalize_cols = ["dist_group"]
celltype_cols = ["Fine.cell.type", "Final.cell.type", "Parent.cell.type", "Grandparent.cell.type"]
# normalize_cols = ["Parent.cell.type", "Grandparent.cell.type", "tile_id", "orig.ident", "dist_group"]

# separate df for each celltype_col
for celltype_col in celltype_cols:
    print(celltype_col)
    
    all_pct_dfs[celltype_col] = {}

    # separate df for each col to normalize by
    for normalize_col in normalize_cols:
    
        # set key to normalize_col
        all_pct_dfs[celltype_col][normalize_col] = {}
        
        print("Normalizing by", normalize_col)
        
        # for each distance and context
        for context, dist_col in dist_cols.items():
            
            print(" > Context:", context)
            
            # get only cols we need
            if normalize_col in ["orig.ident", "dist_group"]:
                subset = total_metadata[['orig.ident', celltype_col, dist_col]]
            else:
                subset = total_metadata[['orig.ident', celltype_col, normalize_col, dist_col]]
        
            # standardize dist col name
            subset = subset.rename(columns={dist_col: 'dist_group'})
        
            # count = count of cells grouped by sample, dist_group, and fine cell type
            if normalize_col in ["orig.ident", "dist_group"]:
                counts = subset.groupby(['orig.ident', 'dist_group', celltype_col]).size().reset_index(name='count')    
            else:
                counts = subset.groupby(['orig.ident', 'dist_group', celltype_col, normalize_col]).size().reset_index(name='count')
            
            # total_count = count of cells grouped by sample, chosen col to normalize by
            if normalize_col == "orig.ident":
                total_per_group = subset.groupby(['orig.ident']).size().reset_index(name='total_count')
            else:
                total_per_group = subset.groupby(['orig.ident', normalize_col]).size().reset_index(name='total_count')
    
            # merge counts and total_counts, then get pct
            merged = counts.merge(total_per_group, on=['orig.ident', normalize_col])
            merged['pct'] = merged['count'] / merged['total_count'] * 100
    
            # Add the Response column to merged_df by merging on orig.ident
            response_df = total_metadata[['orig.ident', 'Response']].drop_duplicates(subset='orig.ident')
            merged = merged.merge(response_df, on='orig.ident', how='left')
        
            # store under this normalize_col and context
            all_pct_dfs[celltype_col][normalize_col][context] = merged
    
    print(" >> Saved to all_pct_dfs")

Fine.cell.type
Normalizing by dist_group
 > Context: Tumor
 > Context: Cycling_Tumor
 > Context: CD8_T
 > Context: Macrophage_CD163pos
 > Context: Activated_CD8_T
 > Context: Stromal_Undefined_Unknown
 > Context: Cytotoxic_NK
 > Context: Neutrophil
 > Context: B
 > Context: Treg
 > Context: Activated_T
 > Context: Exhausted_CD8
 > Context: Activated_Cytotoxic_NK
 > Context: T
 > Context: CD4_T
 > Context: Activated_CD4_T
 > Context: Th1
 > Context: Activated_B
 > Context: Activated_Exhausted_CD8
 > Context: Endothelial
 > Context: Activated_Neutrophil
 > Context: Activated_Treg
 > Context: Macrophage_CD163neg
 > Context: Activated_Th1
 > Context: Activated_Macrophage_CD163pos
 > Context: Activated_Macrophage_CD163neg
 >> Saved to all_pct_dfs
Final.cell.type
Normalizing by dist_group
 > Context: Tumor
 > Context: Cycling_Tumor
 > Context: CD8_T
 > Context: Macrophage_CD163pos
 > Context: Activated_CD8_T
 > Context: Stromal_Undefined_Unknown
 > Context: Cytotoxic_NK
 > Context: Neutrophi

In [111]:
for celltype_col, normalize_dict in all_pct_dfs.items():
    print(f"\nCell type level: {celltype_col}")
    for normalize_col, context_dict in normalize_dict.items():
        print(f"  Normalized by: {normalize_col}")
        for context, df in context_dict.items():
            print(f"    └── Context: {context:<15} → DataFrame shape: {df.shape}")


Cell type level: Fine.cell.type
  Normalized by: dist_group
    └── Context: Tumor           → DataFrame shape: (1010, 7)
    └── Context: Cycling_Tumor   → DataFrame shape: (1040, 7)
    └── Context: CD8_T           → DataFrame shape: (1011, 7)
    └── Context: Macrophage_CD163pos → DataFrame shape: (1000, 7)
    └── Context: Activated_CD8_T → DataFrame shape: (1012, 7)
    └── Context: Stromal_Undefined_Unknown → DataFrame shape: (967, 7)
    └── Context: Cytotoxic_NK    → DataFrame shape: (1005, 7)
    └── Context: Neutrophil      → DataFrame shape: (1021, 7)
    └── Context: B               → DataFrame shape: (1019, 7)
    └── Context: Treg            → DataFrame shape: (996, 7)
    └── Context: Activated_T     → DataFrame shape: (1023, 7)
    └── Context: Exhausted_CD8   → DataFrame shape: (1005, 7)
    └── Context: Activated_Cytotoxic_NK → DataFrame shape: (1028, 7)
    └── Context: T               → DataFrame shape: (1003, 7)
    └── Context: CD4_T           → DataFrame shape: 

In [114]:
all_pct_dfs["Fine.cell.type"]["dist_group"]["Tumor"]

,orig.ident,dist_group,Fine.cell.type,count,total_count,pct,Response
0,00438,Far,Activated_B,14,9633,0.145334,Yes
1,00438,Far,Activated_CD4_T,757,9633,7.858403,Yes
2,00438,Far,Activated_CD8_T,126,9633,1.308004,Yes
3,00438,Far,Activated_Cytotoxic_NK,98,9633,1.017336,Yes
4,00438,Far,Activated_Exhausted_CD8,22,9633,0.228382,Yes
...,...,...,...,...,...,...,...
1005,49411,Peri,Stromal_Undefined_Unknown,1978,5302,37.306677,No
1006,49411,Peri,T,11,5302,0.207469,No
1007,49411,Peri,Th1,12,5302,0.226330,No
1008,49411,Peri,Treg,13,5302,0.245190,No


In [ ]:
for context, df in all_pct_dfs["tile_id"].items():
    # Group by sample, cell type, and dist group — aggregate median pct
    median_df = df.groupby(['orig.ident', 'Fine.cell.type', 'dist_group'])['pct'].median().reset_index()

    # Add response column back
    response_df = total_metadata[['orig.ident', 'Response']].drop_duplicates(subset='orig.ident')
    median_df = median_df.merge(response_df, on='orig.ident', how='left')

    # Replace in dict
    # all_pct_dfs["tile_id"][context] = median_df

In [114]:
all_pct_dfs["tile_id"]["Tumor"]

,orig.ident,Fine.cell.type,dist_group,pct,Response
0,438,Activated_B,Far,0.595238,Yes
1,438,Activated_B,Moderately_near,0.597020,Yes
2,438,Activated_B,Near,0.591716,Yes
3,438,Activated_CD4_T,Far,0.645161,Yes
4,438,Activated_CD4_T,Moderately_near,1.069519,Yes
...,...,...,...,...,...
1009,49411,Th1,Near,0.990099,No
1010,49411,Treg,Far,0.434783,No
1011,49411,Treg,Moderately_near,0.595238,No
1012,49411,Treg,Near,0.617378,No


In [115]:
for normalize_col, context_dict in all_pct_dfs.items():
    print(f"\nNormalization level: {normalize_col}")
    for context, df in context_dict.items():
        print(f"  └── Context: {context} → DataFrame shape: {df.shape}")


Normalization level: Parent.cell.type
  └── Context: Macrophage_CD163pos → DataFrame shape: (1002, 8)
  └── Context: Th1 → DataFrame shape: (1037, 8)
  └── Context: Stromal_Undefined_Unknown → DataFrame shape: (978, 8)
  └── Context: Tumor → DataFrame shape: (1014, 8)
  └── Context: CD8_T → DataFrame shape: (1008, 8)
  └── Context: Endothelial → DataFrame shape: (989, 8)
  └── Context: CD4_T → DataFrame shape: (988, 8)
  └── Context: Cycling_Tumor → DataFrame shape: (1038, 8)
  └── Context: Macrophage_CD163neg → DataFrame shape: (1000, 8)
  └── Context: Cytotoxic_NK → DataFrame shape: (1008, 8)
  └── Context: Exhausted_CD8 → DataFrame shape: (1010, 8)
  └── Context: Activated_Macrophage_CD163neg → DataFrame shape: (1027, 8)
  └── Context: Neutrophil → DataFrame shape: (1034, 8)
  └── Context: Activated_CD4_T → DataFrame shape: (1003, 8)
  └── Context: T → DataFrame shape: (1013, 8)
  └── Context: Treg → DataFrame shape: (1003, 8)
  └── Context: Activated_Neutrophil → DataFrame shape: 

In [115]:
import json 

def make_json_serializable(obj):
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient='records')
    elif isinstance(obj, dict):
        return {k: make_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_json_serializable(v) for v in obj]
    else:
        return obj

# convert all recursively
all_pct_dfs_serializable = make_json_serializable(all_pct_dfs)

# save
with open('/gpfs/home/yb2612/yb2612_fenyo/data/seurat_objects/plots/cell_proportion_plots/dist_group_dfs.json', 'w') as f:
    json.dump(all_pct_dfs_serializable, f, indent=4)